In [ ]:
from pathlib import Path
from copy import deepcopy
import csv, gzip, hashlib, inspect, json, os, platform, sys, time, traceback
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import scipy.special as special
from scipy.optimize import minimize
from scipy.stats import t as student_t
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss, matthews_corrcoef
from threadpoolctl import threadpool_limits
from joblib import Parallel, delayed, parallel_config

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
CODE_DIR = next((p for p in candidates if (p/"experiment.py").is_file()
                 and (p/"simulation/generate_simulation_data.py").is_file()
                 and (p/"function/DW_BLR.py").is_file()), None)
if CODE_DIR is None:
    raise FileNotFoundError("Start Jupyter in the DWBLR project or its simulation directory")
SIMULATION_DIR = CODE_DIR/"simulation"
sys.path.insert(0, str(CODE_DIR))
import experiment as experiment_module
from simulation import generate_simulation_data as data_module
from experiment import fit_methods
from simulation.parameters import all_parameters, density_parameters
from simulation.generate_simulation_data import read_seed_table, simulation_protocol, generate_task_data, describe_task_data, safe_worker_limit
from simulation.generate_simulation_data import FEATURES, BETA, ENVIRONMENTS, SAMPLE_SIZES, sha256
if Path(experiment_module.__file__).resolve().parent != CODE_DIR or Path(data_module.__file__).resolve().parent != SIMULATION_DIR:
    raise RuntimeError("A module from another project is loaded; restart the kernel")

N_JOBS = 1
N_TEST = 2000
SIZES = list(SAMPLE_SIZES)
REPLICATIONS = list(range(1, 101))
PRIMARY_PREDICTION = "native"
SAVE_POSTSELECTION = True
SAVE_DW_PILOTS = True
RETRY_FAILED = False
STANDARDIZE = dict(experiment_module.STANDARDIZE)
SEED_FILE = SIMULATION_DIR/"seed"/"seed_table.csv"
RESULT_DIR = SIMULATION_DIR/"simulation_results"
METHODS = ["LR", "Lasso", "Elastic Net", "SCAD", "SRDO", "SVI", "DWR", "VBLR", "DW-BLR"]
SELECTORS = ["Lasso", "Elastic Net", "SCAD", "SVI", "VBLR", "DW-BLR"]
OVERRIDES = {}
if PRIMARY_PREDICTION not in {"native", "postselection"}:
    raise ValueError("PRIMARY_PREDICTION must be native or postselection")
if PRIMARY_PREDICTION == "postselection" and not SAVE_POSTSELECTION:
    raise ValueError("Enable SAVE_POSTSELECTION")
if not 1 <= N_JOBS <= safe_worker_limit():
    raise ValueError(f"N_JOBS must not exceed {safe_worker_limit()}")
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)


In [ ]:
all_tasks = read_seed_table(SEED_FILE)
seed_rows = []
for task in all_tasks:
    row = {k: v for k, v in task.items() if k != "test_seeds"}
    row.update({f"test_seed_{r:g}": s for r,s in zip(ENVIRONMENTS, task["test_seeds"])})
    seed_rows.append(row)
display(pd.DataFrame(seed_rows).groupby("n_train").size().rename("repetitions"))
data_protocol = simulation_protocol(SEED_FILE, n_test=N_TEST,
                                    sample_sizes=SIZES, repetitions=REPLICATIONS, n_jobs=N_JOBS)
tasks = data_protocol["tasks"]
RESULT_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(seed_rows).to_csv(RESULT_DIR/"seed_table_all.csv", index=False, encoding="utf-8-sig")


In [ ]:
def plain(value):
    if isinstance(value, dict): return {str(k): plain(v) for k,v in value.items()}
    if isinstance(value, (list, tuple)): return [plain(v) for v in value]
    if isinstance(value, np.ndarray): return plain(value.tolist())
    if isinstance(value, np.generic): return plain(value.item())
    if isinstance(value, float) and not np.isfinite(value): return None
    if isinstance(value, Path): return str(value)
    return value

def save_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name+".tmp")
    temporary.write_text(json.dumps(plain(value), ensure_ascii=False, indent=2, allow_nan=False)+"\n", encoding="utf-8")
    os.replace(temporary, path)

def save_arrays(path, **values):
    path = Path(path)
    payload = {key: dict(dtype=np.asarray(value).dtype.str, shape=list(np.asarray(value).shape),
                        values=plain(np.asarray(value))) for key, value in values.items()}
    temporary = path.with_name(path.name+".tmp")
    encoded = json.dumps(payload, ensure_ascii=False, allow_nan=False, separators=(",", ":")).encode("utf-8")
    temporary.write_bytes(gzip.compress(encoded, compresslevel=3, mtime=0))
    os.replace(temporary, path)


def load_arrays(path):
    payload = json.loads(gzip.decompress(Path(path).read_bytes()).decode("utf-8"))
    return {key: np.asarray(value["values"], dtype=np.dtype(value["dtype"])).reshape(value["shape"])
            for key, value in payload.items()}

def fit_logistic(X, y, support):
    support = np.asarray(support, dtype=int)
    design = np.column_stack([np.ones(len(y)), X[:, support]])
    def fg(theta):
        eta = design@theta
        return float(np.mean(np.logaddexp(0., eta)-y*eta)), design.T@(special.expit(eta)-y)/len(y)
    def hessian(theta):
        p = special.expit(design@theta)
        return (design.T*(p*(1-p)))@design/len(y)
    start = np.zeros(design.shape[1])
    start[0] = np.log(np.clip(np.mean(y), 1e-12, 1-1e-12)/np.clip(1-np.mean(y), 1e-12, 1))
    fitted = minimize(fg, start, jac=True, method="L-BFGS-B",
                      options=dict(maxiter=10000, gtol=1e-10, ftol=1e-14, maxls=50))
    gradient = float(np.max(np.abs(fg(fitted.x)[1])))
    if gradient > 1e-6:
        fitted = minimize(lambda v: fg(v)[0], fitted.x, jac=lambda v: fg(v)[1],
                          hess=hessian, method="trust-exact", options=dict(maxiter=1000, gtol=1e-9))
        gradient = float(np.max(np.abs(fg(fitted.x)[1])))
    if not np.isfinite(fitted.x).all() or gradient > 1e-6:
        raise RuntimeError(f"Logistic refit failed: gradient={gradient}")
    beta = np.zeros(X.shape[1])
    beta[support] = fitted.x[1:]
    return beta, float(fitted.x[0]), dict(converged=True, iterations=int(fitted.nit), gradient_inf=gradient)

def selection_metrics(support, p=30):
    mask = np.zeros(p, dtype=bool)
    mask[np.asarray(support, dtype=int)] = True
    truth = np.arange(p) < 5
    tp, fp = int(np.sum(mask & truth)), int(np.sum(mask & ~truth))
    fn, tn = 5-tp, p-5-fp
    return dict(TP=tp, FP=fp, FN=fn, TN=tn, TPR=tp/5, FDR=fp/max(tp+fp,1),
                MCC=float(matthews_corrcoef(truth, mask)), model_size=int(mask.sum()),
                exact_support=bool(np.array_equal(mask, truth)), FPR=fp/(p-5), FNR=fn/5)

def weight_metrics(X, weights):
    w = np.asarray(weights, dtype=float)
    if w.shape != (len(X),) or not np.isfinite(w).all() or np.any(w < 0) or w.sum() <= 0:
        raise ValueError("Invalid weights")
    q = w/w.sum()
    centered = X-q@X
    cov = (centered*q[:, None]).T@centered
    scale = np.sqrt(np.diag(cov))
    if np.any(scale <= 0): raise ValueError("Degenerate weighted covariance")
    corr = cov/np.outer(scale, scale)
    ess = float(1/(q@q))
    return dict(ESS=ess, ESS_rate=ess/len(w), max_weight_share=float(q.max()),
                SV_corr_norm=float(np.linalg.norm(corr[:5,5:], ord="fro")),
                SS_corr_norm=float(np.linalg.norm(corr[:5,:5]-np.eye(5), ord="fro"))), corr

def prediction_metrics(y, p):
    p = np.asarray(p, dtype=float)
    if p.shape != y.shape or not np.isfinite(p).all(): raise ValueError("Invalid probabilities")
    return dict(Log_loss=float(log_loss(y, np.clip(p,1e-12,1-1e-12), labels=[0,1])),
                AUROC=float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None,
                Brier=float(brier_score_loss(y,p)))

def coefficient_metrics(beta):
    return dict(beta_V_RMSE=float(np.sqrt(np.mean(beta[5:]**2))),
                beta_V_L1=float(np.abs(beta[5:]).sum()),
                beta_S_RMSE=float(np.sqrt(np.mean((beta[:5]-BETA[:5])**2))))

def effective_options(n):
    cfg = all_parameters(n,30)
    for method, update in OVERRIDES.items():
        cfg[method].update(deepcopy(update))
    return cfg

source_files = [*CODE_DIR.glob("*.py"), *(CODE_DIR/"function").glob("*.py"),
                *(CODE_DIR/"_backends").glob("*.py"), *SIMULATION_DIR.glob("*.py")]
source_hashes = {p.relative_to(CODE_DIR).as_posix(): sha256(p) for p in sorted(source_files)}
notebook_file = json.loads((SIMULATION_DIR/"simulation_comparison.ipynb").read_text(encoding="utf-8"))
notebook_sources = ["".join(c["source"]) if isinstance(c["source"],list) else c["source"] for c in notebook_file["cells"] if c["cell_type"]=="code"]
fit_sources = [s for s in notebook_sources if "\ndef fit_logistic(" in "\n"+s or "\ndef fit_one_method(" in "\n"+s]
if len(fit_sources)!=2:raise ValueError("Cannot locate the fitting functions in the notebook")
notebook_fit_hash = hashlib.sha256("\n".join(fit_sources).encode()).hexdigest()
data_manifest_path = RESULT_DIR/"data_manifest.json"
if data_manifest_path.exists() and json.loads(data_manifest_path.read_text(encoding="utf-8")) != data_protocol:
    raise ValueError("Data protocol changed; choose a new RESULT_DIR")
save_json(data_manifest_path, data_protocol)
fit_protocol = dict(schema_version=2, data_storage="memory", array_storage="json.gz",
                    data_manifest_sha256=sha256(data_manifest_path),
                    seed_table_sha256=sha256(SEED_FILE), methods=METHODS, standardize=STANDARDIZE,
                    configs={n: effective_options(n) for n in SIZES}, source_sha256=source_hashes,
                    save_postselection=SAVE_POSTSELECTION, save_dw_pilots=SAVE_DW_PILOTS,notebook_fit_code_sha256=notebook_fit_hash)
FINGERPRINT = hashlib.sha256(json.dumps(plain(fit_protocol), sort_keys=True).encode()).hexdigest()
protocol_path = RESULT_DIR/"fit_protocol.json"
if protocol_path.exists() and json.loads(protocol_path.read_text(encoding="utf-8")) != plain(fit_protocol):
    raise ValueError("Model settings or code have changed; use a new RESULT_DIR")
save_json(protocol_path, fit_protocol)
from importlib.metadata import version
save_json(RESULT_DIR/"runtime_versions.json", dict(python=platform.python_version(),
          executable=sys.executable, platform=platform.platform(),
          packages={name:version(name) for name in ['numpy', 'scipy', 'pandas', 'openpyxl', 'scikit-learn', 'torch', 'matplotlib', 'joblib', 'psutil', 'threadpoolctl', 'ipykernel', 'jupyterlab', 'nbformat', 'nbclient']}))


In [ ]:
def model_status(model):
    result = getattr(model, "result_", None)
    info = dict(converged=None, iterations=None)
    if hasattr(model, "final_model_") and model.final_model_ is not None:
        info.update(converged=bool(model.final_model_.converged_),
                    iterations=len(model.final_model_.history_), valid_pilots=model.n_valid_subsamples_,
                    failed_pilots=model.n_subsamples-model.n_valid_subsamples_)
    elif result is not None and hasattr(result, "converged"):
        info.update(converged=bool(result.converged),
                    iterations=int(getattr(result,"n_iterations",getattr(result,"n_iter",0))))
    elif result is not None and hasattr(result,"final_fit"):
        info.update(converged=bool(result.final_fit.converged), iterations=int(result.final_fit.iterations))
    elif result is not None and hasattr(result,"final_logistic_info"):
        detail = result.final_logistic_info
        info.update(converged=bool(detail.get("success",False)), iterations=detail.get("n_iter"))
        info["selector_epochs"] = len(result.selector_history)
    return info

def checkpoint_folder(task, method):
    name = method.replace(" ","_").replace("-","_")
    return RESULT_DIR/f"n{task['n_train']}"/f"rep_{task['replication']:03d}"/name

def load_checkpoint(folder):
    path = folder/"result.json"
    if not path.exists(): return None
    row = json.loads(path.read_text(encoding="utf-8"))
    if row["fingerprint"] != FINGERPRINT: raise ValueError(f"Checkpoint configuration mismatch: {folder}")
    for name,digest in row.get("sha256",{}).items():
        if sha256(folder/name) != digest: raise ValueError(f"Checkpoint hash mismatch: {folder/name}")
    return row

def fit_one_method(task, method, train, tests):
    folder = checkpoint_folder(task, method)
    folder.mkdir(parents=True, exist_ok=True)
    cached = load_checkpoint(folder)
    if cached is not None and (cached["status"]=="ok" or not RETRY_FAILED): return cached
    n, rep = task["n_train"], task["replication"]
    base = dict(n_train=n, replication=rep, method=method)
    result = dict(**base, fingerprint=FINGERPRINT, task=task, status="error", metrics=None, environments=[])
    started = time.perf_counter()
    try:
        X, y = train["X"], train["y"]
        model = fitted = None
        cfg = {}
        extras = {}
        if method == "LR":
            support = np.arange(30)
            beta, intercept, status = fit_logistic(X,y,support)
            probabilities = {r: special.expit(intercept+d["X"]@beta) for r,d in tests.items()}
            weights = np.ones(n)
        else:
            cfg = effective_options(n)[method]
            if method=="DW-BLR": cfg["output_dir"] = None
            if method=="SVI" and STANDARDIZE["SVI"]==STANDARDIZE["SRDO"] and "precomputed_srdo_weights" not in cfg:
                source = checkpoint_folder(task,"SRDO")
                stored = load_checkpoint(source)
                density_cfg = density_parameters(n)
                srdo_cfg = {k:effective_options(n)["SRDO"][k] for k in density_cfg}
                svi_cfg = dict(density_cfg)
                svi_cfg.update(cfg.get("srdo_params") or {})
                if stored is not None and stored["status"]=="ok" and srdo_cfg==svi_cfg:
                    arrays = load_arrays(source/"state.json.gz")
                    cfg["precomputed_srdo_weights"] = arrays["density_weights"].copy()
            fitted = fit_methods(X,y,methods=[method],params={method:cfg},standardize=STANDARDIZE,feature_names=FEATURES)[method]
            model = fitted.model
            beta, intercept = fitted.coefficients()
            support = model.get_selected_features() if hasattr(model,"get_selected_features") else np.arange(30)
            weights = model.get_weights() if hasattr(model,"get_weights") else np.ones(n)
            status = model_status(model)
            probabilities = {r: fitted.predict_proba(d["X"])[:,1] for r,d in tests.items()}
            if method=="SRDO": extras["density_weights"] = model.get_srdo_weights()
            if method=="DWR": extras["decorrelation_weights"] = model.result_.dwr_weights.copy()
            if method=="SVI":
                extras["gate_probability"] = model.get_selection_probabilities()
                status.update(lambda_stg=model.lambda_stg_, selector_ESS=model.ESS_)
            if method in {"VBLR","DW-BLR"}: extras["PIP"] = model.get_pip()
            if method=="VBLR": status["final_shared_prior"] = float(model.result_.hyper.pi0)
            if method=="DW-BLR":
                extras.update(initial_prior=model.get_initial_prior(), final_prior=model.get_feature_prior())
                save_json(folder/"pilot_status.json",model.pilot_status_)
                if SAVE_DW_PILOTS:
                    extras["subsample_indices"] = model.subsample_indices_.astype(np.int32)
                    extras["pilot_PIP"] = np.stack([v.pip if v is not None else np.full(30,np.nan) for v in model.pilot_results_])
                    extras["pilot_valid"] = np.asarray([s["converged"] for s in model.pilot_status_])
        wm, corr = weight_metrics(X,weights)
        metric_row = dict(**base, **selection_metrics(support), **wm,
                          selection_available=method in SELECTORS,
                          selected_variables=[FEATURES[int(j)] for j in support], **status)
        metric_row.update({f"{k}_native":v for k,v in coefficient_metrics(beta).items()})
        metric_row["final_fit_ESS_native"] = wm["ESS"] if method in {"SRDO","DWR","DW-BLR"} else n
        environment_rows = []
        for r, data in tests.items():
            environment_rows.append(dict(**base,r_test=r,prediction="native",status="ok",**prediction_metrics(data["y"],probabilities[r])))
        saved_arrays = dict(beta_native=beta, intercept_native=intercept, support=np.asarray(support,dtype=int),
                            weights=weights, correlation=corr, environments=np.asarray(ENVIRONMENTS),
                            predictions_native=np.stack([probabilities[r] for r in ENVIRONMENTS]), **extras)
        if SAVE_POSTSELECTION and method in SELECTORS:
            try:
                post_beta, post_intercept, post_status = fit_logistic(X,y,support)
                metric_row.update({f"{k}_postselection":v for k,v in coefficient_metrics(post_beta).items()})
                saved_arrays.update(beta_postselection=post_beta,intercept_postselection=post_intercept,
                    predictions_postselection=np.stack([special.expit(post_intercept+tests[r]["X"]@post_beta) for r in ENVIRONMENTS]))
                for i,r in enumerate(ENVIRONMENTS):
                    environment_rows.append(dict(**base,r_test=r,prediction="postselection",status="ok",
                        **prediction_metrics(tests[r]["y"],saved_arrays["predictions_postselection"][i])))
                metric_row["postselection_converged"] = post_status["converged"]
            except Exception as exc:
                metric_row["postselection_error"] = f"{type(exc).__name__}: {exc}"
                for r in ENVIRONMENTS:
                    environment_rows.append(dict(**base,r_test=r,prediction="postselection",status="error",
                                                 Log_loss=None, AUROC=None, Brier=None))
        save_arrays(folder/"state.json.gz",**saved_arrays)
        if model is not None:
            history = getattr(model,"history_",None)
            if history is None and hasattr(model.result_,"selector_history"):history=model.result_.selector_history
            if history is not None:save_json(folder/"history.json",history)
        result.update(status="ok",metrics=metric_row,environments=environment_rows,
                      sha256={p.name:sha256(p) for p in folder.iterdir() if p.is_file() and (p.suffix==".json" or p.name=="state.json.gz") and p.name!="result.json"})
    except Exception as exc:
        result.update(error=f"{type(exc).__name__}: {exc}",traceback=traceback.format_exc(),sha256={})
        if hasattr(exc,"model") and hasattr(exc.model,"pilot_status_"):
            save_json(folder/"failed_pilot_status.json",exc.model.pilot_status_)
        result["environments"] = []
        for variant in (["native","postselection"] if SAVE_POSTSELECTION and method in SELECTORS else ["native"]):
            for r in ENVIRONMENTS:
                result["environments"].append(dict(**base,r_test=r,prediction=variant,status="error",Log_loss=None,AUROC=None,Brier=None))
    result["elapsed_seconds"] = time.perf_counter()-started
    save_json(folder/"result.json",result)
    return result

def run_one_task(task):
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    with threadpool_limits(limits=1):
        train, tests = generate_task_data(task,n_test=N_TEST)
        raw_metrics, _ = weight_metrics(train["X"],np.ones(task["n_train"]))
        folder = RESULT_DIR/f"n{task['n_train']}"/f"rep_{task['replication']:03d}"
        folder.mkdir(parents=True,exist_ok=True)
        save_json(folder/"data_record.json",describe_task_data(task,train,tests))
        save_json(folder/"original_data_metrics.json",dict(n_train=task["n_train"],replication=task["replication"],**raw_metrics))
        records = [fit_one_method(task,m,train,tests) for m in METHODS]
    return dict(n_train=task["n_train"],replication=task["replication"],
                ok=sum(r["status"]=="ok" for r in records),failed=sum(r["status"]!="ok" for r in records))


In [ ]:
lock_path = RESULT_DIR/"running.lock"
handle = os.open(lock_path,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
os.write(handle,json.dumps(dict(pid=os.getpid(),fingerprint=FINGERPRINT)).encode())
os.close(handle)
try:
    if N_JOBS==1:
        progress=[]
        for i,task in enumerate(tasks,1):
            progress.append(run_one_task(task))
            print(f"{i}/{len(tasks)}: n={task['n_train']}, rep={task['replication']}, {progress[-1]}",flush=True)
    else:
        with parallel_config(backend="loky",inner_max_num_threads=1):
            progress=Parallel(n_jobs=N_JOBS,verbose=10)(delayed(run_one_task)(t) for t in tasks)
finally:
    lock_path.unlink(missing_ok=True)
display(pd.DataFrame(progress).groupby("n_train")[["ok","failed"]].sum())


In [ ]:
records=[load_checkpoint(checkpoint_folder(t,m)) for t in tasks for m in METHODS]
if any(r is None for r in records):raise RuntimeError("Missing method checkpoint")
environment=pd.DataFrame([e for r in records for e in r["environments"]])
selection=pd.DataFrame([dict(**r["metrics"],fit_status="ok") if r["status"]=="ok" else
    dict(n_train=r["n_train"],replication=r["replication"],method=r["method"],fit_status="error") for r in records])
failures=pd.DataFrame([{k:r.get(k) for k in ("n_train","replication","method","error")} for r in records if r["status"]!="ok"],columns=["n_train","replication","method","error"])
diagnostics=pd.DataFrame([dict(n_train=r["n_train"],replication=r["replication"],method=r["method"],
    status=r["status"],elapsed_seconds=r["elapsed_seconds"],**({k:r["metrics"].get(k) for k in ("converged","iterations","valid_pilots","failed_pilots","selector_epochs","lambda_stg","selector_ESS")} if r["status"]=="ok" else {})) for r in records])
if not any(r["status"]=="ok" for r in records):
    failures.to_csv(RESULT_DIR/"failures.csv",index=False,encoding="utf-8-sig")
    raise RuntimeError("All methods failed; inspect result.json")
for metric in ("beta_V_RMSE","beta_V_L1","beta_S_RMSE"):
    selection[metric]=[row.get(f"{metric}_postselection") if PRIMARY_PREDICTION=="postselection" and row["method"] in SELECTORS else row.get(f"{metric}_native") for row in selection.to_dict("records")]
selection["final_fit_ESS"]=[row["n_train"] if PRIMARY_PREDICTION=="postselection" and row["method"] in SELECTORS else row["final_fit_ESS_native"] for row in selection.to_dict("records")]
selection["primary_prediction"]=["postselection" if PRIMARY_PREDICTION=="postselection" and m in SELECTORS else "native" for m in selection["method"]]
primary=environment[environment.apply(lambda row:row["prediction"]==("postselection" if PRIMARY_PREDICTION=="postselection" and row["method"] in SELECTORS else "native"),axis=1)].copy()
expected_tasks=len(tasks)
if len(records)!=expected_tasks*len(METHODS):raise RuntimeError("Incorrect number of method records")
if len(primary)!=expected_tasks*len(METHODS)*len(ENVIRONMENTS):raise RuntimeError("Incorrect number of test-environment records")
for name,frame in [("environment_per_rep",environment),("selection_per_rep",selection),("failures",failures),("diagnostics",diagnostics),("primary_environment_per_rep",primary)]:
    frame.to_csv(RESULT_DIR/f"{name}.csv",index=False,encoding="utf-8-sig")

def summarize(frame,groups,metrics):
    output=[]
    for keys,block in frame.groupby(groups,dropna=False):
        if not isinstance(keys,tuple):keys=(keys,)
        base=dict(zip(groups,keys))
        for metric in metrics:
            values=pd.to_numeric(block[metric],errors="coerce").to_numpy(dtype=float)
            values=values[np.isfinite(values)]
            count=len(values);mean=float(values.mean()) if count else np.nan
            sd=float(values.std(ddof=1)) if count>1 else np.nan
            se=sd/np.sqrt(count) if count>1 else np.nan
            half=float(student_t.ppf(.975,count-1)*se) if count>1 else np.nan
            output.append(dict(**base,metric=metric,n_valid=count,n_records=len(block),mean=mean,SD=sd,MCSE=se,
                               lower=mean-half,upper=mean+half))
    return pd.DataFrame(output)

prediction_metrics_names=["Log_loss","AUROC","Brier"]
environment_summary=summarize(primary,["n_train","method","r_test"],prediction_metrics_names)
grouped=primary.groupby(["n_train","replication","method"],as_index=False)[prediction_metrics_names].agg(lambda s:np.mean(s.to_numpy(dtype=float)) if s.notna().sum()==8 else np.nan)
overall_summary=summarize(grouped,["n_train","method"],prediction_metrics_names)


def mean_sd_table(summary, groups):
    metrics = ["Log_loss", "Brier", "AUROC"]
    statistics = ["mean", "SD", "n_valid"]
    wide = summary.pivot(index=groups, columns="metric", values=statistics)
    wide.columns = [f"{metric}_{statistic}" for statistic, metric in wide.columns]
    columns = groups + [f"{metric}_{statistic}" for metric in metrics for statistic in statistics]
    return wide.reset_index().reindex(columns=columns)

environment_mean_sd=mean_sd_table(environment_summary,["n_train","r_test","method"])
overall_8env_mean_sd=mean_sd_table(overall_summary,["n_train","method"])
environment_mean_sd.to_csv(RESULT_DIR/"environment_mean_sd.csv",index=False,encoding="utf-8-sig")
overall_8env_mean_sd.to_csv(RESULT_DIR/"overall_8env_mean_sd.csv",index=False,encoding="utf-8-sig")
selection_fields=["TPR","FDR","MCC","model_size","beta_V_RMSE","beta_V_L1","beta_S_RMSE","ESS","ESS_rate","SV_corr_norm","SS_corr_norm","max_weight_share","exact_support","final_fit_ESS"]
selection_summary=summarize(selection,["n_train","method"],selection_fields)
paired=[]
for n in SIZES:
    for metric in prediction_metrics_names:
        wide=grouped[grouped.n_train==n].pivot(index="replication",columns="method",values=metric)
        if "DW-BLR" not in wide:continue
        for baseline in METHODS:
            if baseline=="DW-BLR" or baseline not in wide:continue
            diff=(wide["DW-BLR"]-wide[baseline]).dropna().to_numpy()
            count=len(diff);mean=float(diff.mean()) if count else np.nan
            se=float(diff.std(ddof=1)/np.sqrt(count)) if count>1 else np.nan
            half=float(student_t.ppf(.975,count-1)*se) if count>1 else np.nan
            beneficial=diff if metric=="AUROC" else -diff
            paired.append(dict(n_train=n,metric=metric,baseline=baseline,n_pairs=count,
                difference_DW_minus_baseline=mean,MCSE=se,lower=mean-half,upper=mean+half,
                win_rate=float(np.mean(beneficial>1e-10)) if count else np.nan,
                tie_rate=float(np.mean(np.abs(beneficial)<=1e-10)) if count else np.nan))
paired=pd.DataFrame(paired)
for name,frame in [("environment_summary",environment_summary),("overall_per_rep",grouped),("overall_summary",overall_summary),
                   ("selection_summary",selection_summary),("paired_comparisons",paired)]:
    frame.to_csv(RESULT_DIR/f"{name}.csv",index=False,encoding="utf-8-sig")
save_json(RESULT_DIR/"analysis_config.json",dict(primary_prediction=PRIMARY_PREDICTION))
display(overall_summary)
display(selection_summary)
display(diagnostics.groupby(["n_train","method","status"],dropna=False).size().rename("count"))
display(failures)


In [ ]:
COLORS={"LR":"#66758A","Lasso":"#6A5C95","Elastic Net":"#5E99AB",
        "SCAD":"#8FA2D6","DWR":"#7BA98D","SRDO":"#B66E9C","SVI":"#D99AAE","VBLR":"#BFA164","DW-BLR":"#D38167"}
STYLES={"LR":(0,(7,3)),"Lasso":(0,(3,2)),"Elastic Net":"-.","SCAD":":",
        "DWR":(0,(8,2,1,2)),"SRDO":(0,(5,2,1,2,1,2)),"SVI":(0,(9,3)),"VBLR":(0,(2,1,2,3)),"DW-BLR":"-"}
MARKERS=dict(zip(METHODS,["p","s","^","v","X","h","P","D","o"]))
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"axes.spines.top":False,
    "axes.spines.right":False,"axes.axisbelow":True,"pdf.fonttype":42,"ps.fonttype":42})
FIGURE_DIR=RESULT_DIR/"figures"
FIGURE_DIR.mkdir(exist_ok=True)
fig,axes=plt.subplots(len(SIZES),3,figsize=(19,4.3*len(SIZES)),squeeze=False)
x=np.arange(len(ENVIRONMENTS))
for row,n in enumerate(SIZES):
    for col,(metric,label) in enumerate([("Log_loss","Log-loss ↓"),("Brier","Brier score ↓"),("AUROC","AUROC ↑")]):
        ax=axes[row,col]
        for method in METHODS:
            block=environment_summary[(environment_summary.n_train==n)&(environment_summary.method==method)&(environment_summary.metric==metric)].set_index("r_test").reindex(ENVIRONMENTS)
            ax.plot(x,block["mean"],color=COLORS[method],linestyle=STYLES[method],marker=MARKERS[method],markersize=4,linewidth=1.7,label=method)
            ax.fill_between(x,block["lower"].to_numpy(dtype=float),block["upper"].to_numpy(dtype=float),color=COLORS[method],alpha=.10,linewidth=0)
        ax.set_xticks(x,[f"{r:g}" for r in ENVIRONMENTS])
        ax.set_xlabel(r"$r$ on test data")
        ax.set_ylabel((f"n = {n:,}\n" if col==0 else "")+label.replace(" ↓","").replace(" ↑",""))
        ax.grid(axis="y",color="#dfe5ee",linewidth=.8)
        if row==0:ax.set_title(label,pad=12)
handles=[Line2D([0],[0],color=COLORS[m],linestyle=STYLES[m],marker=MARKERS[m],label=m) for m in METHODS]
fig.legend(handles=handles,loc="lower center",ncol=len(METHODS),fontsize=9,bbox_to_anchor=(.5,.005),frameon=False)
fig.tight_layout(rect=(0,.065,1,1))
fig.savefig(FIGURE_DIR/"cross_environment.png",dpi=300,bbox_inches="tight")
fig.savefig(FIGURE_DIR/"cross_environment.pdf",bbox_inches="tight")
plt.show()

fig,axes=plt.subplots(2,3,figsize=(20,10))
panels=[("TPR","TPR ↑",SELECTORS),("FDR","FDR ↓",SELECTORS),("MCC","MCC ↑",SELECTORS),
        ("model_size","Model size",SELECTORS),("beta_V_RMSE",r"$\beta_V$ RMSE ↓",METHODS),
        ("ESS","ESS ↑",["SRDO","SVI","DWR","DW-BLR"])]
for index,(ax,(metric,title,methods)) in enumerate(zip(axes.flat,panels)):
    centers=np.arange(len(SIZES));width=.82/len(methods)
    for k,method in enumerate(methods):
        block=selection_summary[(selection_summary.method==method)&(selection_summary.metric==metric)].set_index("n_train").reindex(SIZES)
        means=block["mean"].to_numpy(dtype=float)
        errors=(block["upper"]-block["mean"]).to_numpy(dtype=float)
        positions=centers+(k-(len(methods)-1)/2)*width
        ax.bar(positions,means,width,color=COLORS[method],alpha=1,edgecolor="white",linewidth=.6)
        valid=np.isfinite(errors)&np.isfinite(means)
        if valid.any():ax.errorbar(positions[valid],means[valid],yerr=errors[valid],fmt="none",ecolor="#424954",elinewidth=.9,capsize=2)
    ax.set_xticks(centers,[f"{n:,}" for n in SIZES]);ax.set_xlabel("Sample size (n)")
    ax.set_ylabel("Mean (95% CI)");ax.set_title(title,pad=12)
    ax.text(-.12,1.04,chr(97+index),transform=ax.transAxes,fontsize=15,fontweight="bold")
    ax.grid(axis="y",color="#dfe5ee",linewidth=.8)
    if metric=="model_size":ax.axhline(5,color="#66758A",linestyle="--",linewidth=1)
bar_legend_methods=METHODS.copy()
handles=[Patch(facecolor=COLORS[m],edgecolor="white",label=m) for m in bar_legend_methods]
fig.legend(handles=handles,loc="lower center",ncol=len(bar_legend_methods),fontsize=9,bbox_to_anchor=(.5,.005),frameon=False)
fig.tight_layout(rect=(0,.065,1,1))
fig.savefig(FIGURE_DIR/"selection_estimation_ESS.png",dpi=300,bbox_inches="tight")
fig.savefig(FIGURE_DIR/"selection_estimation_ESS.pdf",bbox_inches="tight")
plt.show()


In [ ]:
original=pd.DataFrame([json.loads((RESULT_DIR/f"n{t['n_train']}"/f"rep_{t['replication']:03d}"/"original_data_metrics.json").read_text(encoding="utf-8")) for t in tasks])
original["method"]="Origin"
norm_data=pd.concat([original,selection[selection.method.isin(["SRDO","DWR","DW-BLR"])]],ignore_index=True)
norm_summary=summarize(norm_data,["n_train","method"],["SV_corr_norm","SS_corr_norm"])
norm_data.to_csv(RESULT_DIR/"correlation_norm_per_rep.csv",index=False,encoding="utf-8-sig")
norm_summary.to_csv(RESULT_DIR/"correlation_norm_summary.csv",index=False,encoding="utf-8-sig")
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
norm_methods=["Origin","SRDO","DWR","DW-BLR"]
for ax,(metric,title) in zip(axes,[("SV_corr_norm","Stable-spurious correlation norm ↓"),("SS_corr_norm","Stable-stable off-diagonal correlation norm")]):
    for k,m in enumerate(norm_methods):
        block=norm_summary[(norm_summary.method==m)&(norm_summary.metric==metric)].set_index("n_train").reindex(SIZES)
        errors=(block["upper"]-block["mean"]).to_numpy(dtype=float)
        positions=np.arange(len(SIZES))+(k-1.5)*.19
        means=block["mean"].to_numpy(dtype=float)
        ax.bar(positions,means,.19,color="#A9B2BF" if m=="Origin" else COLORS[m],edgecolor="white",linewidth=.6,label=m)
        valid=np.isfinite(errors)&np.isfinite(means)
        if valid.any():ax.errorbar(positions[valid],means[valid],yerr=errors[valid],fmt="none",ecolor="#424954",elinewidth=.9,capsize=2)
    ax.set_xticks(np.arange(len(SIZES)),[f"{n:,}" for n in SIZES]);ax.set_xlabel("Sample size (n)")
    ax.set_ylabel("Mean (95% CI)");ax.set_title(title);ax.grid(axis="y",color="#dfe5ee")
fig.legend(*axes[0].get_legend_handles_labels(),ncol=4,loc="lower center",frameon=False)
fig.tight_layout(rect=(0,.08,1,1))
fig.savefig(FIGURE_DIR/"correlation_norms.png",dpi=300,bbox_inches="tight")
fig.savefig(FIGURE_DIR/"correlation_norms.pdf",bbox_inches="tight")
plt.show()

feature_rows=[]
for task in tasks:
    for method in METHODS:
        folder=checkpoint_folder(task,method);record=load_checkpoint(folder)
        if record["status"]!="ok":continue
        state=load_arrays(folder/"state.json.gz")
        selected=set(int(j) for j in state["support"])
        for j,gene in enumerate(FEATURES):
            feature_rows.append(dict(n_train=task["n_train"],replication=task["replication"],method=method,
                feature=gene,true_beta=BETA[j],selected=j in selected,beta_native=state["beta_native"][j],
                beta_postselection=state["beta_postselection"][j] if "beta_postselection" in state else None,
                PIP=state["PIP"][j] if "PIP" in state else None,
                gate_probability=state["gate_probability"][j] if "gate_probability" in state else None,
                initial_prior=state["initial_prior"][j] if "initial_prior" in state else None,
                final_prior=state["final_prior"][j] if "final_prior" in state else None))
pd.DataFrame(feature_rows).to_csv(RESULT_DIR/"feature_results.csv",index=False,encoding="utf-8-sig")
save_json(RESULT_DIR/"completion_audit.json",dict(expected_method_records=len(tasks)*len(METHODS),
    actual_method_records=len(records),successful=sum(r["status"]=="ok" for r in records),
    failed=sum(r["status"]!="ok" for r in records),actual_primary_environment_records=len(primary),
    primary_prediction=PRIMARY_PREDICTION,all_repetitions_retained=True,
    seed_table_sha256=sha256(SEED_FILE),data_manifest_sha256=sha256(data_manifest_path)))
display(pd.DataFrame([dict(file=str(p.relative_to(RESULT_DIR)),bytes=p.stat().st_size) for p in sorted(RESULT_DIR.glob("*.csv"))]))
print("Data: generated from seed/seed_table.csv in memory")
print("Results:",RESULT_DIR)
print("Figures:",FIGURE_DIR)
